# Cap 05 — Salida Estructurada

Usamos Pydantic + `with_structured_output` para obtener datos tipados del LLM:
- `NolanFilmAnalysis` como schema de salida
- Nodo LLM con `with_structured_output`
- Cadena de 2 nodos estructurados
- Fallback ante `ValidationError`

**Dominio**: Películas de Christopher Nolan  
**Flujo**: `START → extractor → analista → evaluador → END`

In [ ]:
import os
import json
from pathlib import Path
from typing import Optional, List
from dotenv import load_dotenv

load_dotenv()

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from pydantic import BaseModel, Field, ValidationError
from rich import print as rprint

DATA_PATH = Path("../00_datos/nolan_films.json")
nolan_films = json.loads(DATA_PATH.read_text(encoding="utf-8"))

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), temperature=0)
print(f"LLM: {llm.model_name} ✓")

## 1. El problema del parsing manual

Sin salida estructurada, extraer datos del LLM requiere parsing frágil con regex.

In [ ]:
# ❌ Enfoque frágil: parsing de texto libre
respuesta_texto = llm.invoke([
    HumanMessage(content="Dame el título y año de Inception en formato JSON")
])
print("Respuesta texto libre:")
print(respuesta_texto.content)
print("\nProblema: el formato puede variar, los campos pueden faltar, los tipos son incorrectos")

## 2. La solución: `with_structured_output` + Pydantic

In [ ]:
class NolanFilmAnalysis(BaseModel):
    """Análisis estructurado de una película de Christopher Nolan."""
    film_title: str = Field(description="Título de la película analizada")
    year: Optional[int] = Field(default=None, description="Año de estreno")
    main_themes: List[str] = Field(default_factory=list, description="Temas principales")
    narrative_technique: str = Field(default="", description="Técnica narrativa clave")
    analysis: str = Field(description="Análisis detallado de la consulta")
    connections: List[str] = Field(default_factory=list, description="Conexiones con otras películas de Nolan")
    confianza: float = Field(default=0.0, ge=0.0, le=1.0, description="Confianza del modelo en el análisis (0-1)")

class FilmEvaluation(BaseModel):
    """Evaluación de calidad del análisis."""
    completeness_score: float = Field(ge=0, le=10, description="Puntuación de completitud (0-10)")
    has_specific_examples: bool = Field(description="¿Incluye ejemplos concretos?")
    improvement_suggestion: str = Field(default="", description="Sugerencia de mejora")

print("Modelos Pydantic definidos:")
rprint(NolanFilmAnalysis.model_json_schema())

In [ ]:
# ✅ Enfoque estructurado
structured_llm = llm.with_structured_output(NolanFilmAnalysis)

films_context = json.dumps([{
    "titulo": f["titulo"], "año": f["año"], 
    "temas": f["temas"], "tecnica": f["tecnica_narrativa"]
} for f in nolan_films], ensure_ascii=False)

EXTRACTOR_SYSTEM = f"""Eres un crítico de cine especializado en Christopher Nolan.
Analiza la consulta y extrae información estructurada sobre la película más relevante.

Películas disponibles:
{films_context}"""

resultado: NolanFilmAnalysis = structured_llm.invoke([
    SystemMessage(content=EXTRACTOR_SYSTEM),
    HumanMessage(content="¿Qué técnica narrativa usa Memento y por qué es importante?")
])

print("✅ Resultado estructurado:")
rprint(resultado)
print(f"\nTipo: {type(resultado)}")
print(f"Año (int): {resultado.year}")
print(f"Temas (list): {resultado.main_themes}")

## 3. Estado con Objetos Pydantic

El estado del grafo puede contener objetos Pydantic, no solo tipos primitivos.

In [ ]:
from typing import Optional as Opt

class AnalysisState(MessagesState):
    film_analysis: Opt[NolanFilmAnalysis]
    evaluation: Opt[FilmEvaluation]
    query: str

print("Estado con campos Pydantic definido ✓")

## 4. Cadena de 2 Nodos Estructurados

In [ ]:
structured_extractor = llm.with_structured_output(NolanFilmAnalysis)
structured_evaluator = llm.with_structured_output(FilmEvaluation)

def node_extractor(state: AnalysisState) -> dict:
    """Nodo 1: extrae análisis estructurado de la película."""
    try:
        analysis = structured_extractor.invoke(
            [SystemMessage(content=EXTRACTOR_SYSTEM)] + state["messages"]
        )
        return {
            "film_analysis": analysis,
            "messages": [AIMessage(content=analysis.analysis)]
        }
    except (ValidationError, Exception) as e:
        print(f"  ⚠️ Fallback en extractor: {e}")
        fallback = NolanFilmAnalysis(
            film_title="Película desconocida",
            analysis="No se pudo extraer análisis estructurado."
        )
        return {"film_analysis": fallback, "messages": [AIMessage(content=fallback.analysis)]}

def node_evaluador(state: AnalysisState) -> dict:
    """Nodo 2: evalúa la calidad del análisis anterior."""
    analysis = state.get("film_analysis")
    if not analysis:
        return {}
    
    EVAL_SYSTEM = "Evalúa la calidad del siguiente análisis cinematográfico."
    try:
        evaluation = structured_evaluator.invoke([
            SystemMessage(content=EVAL_SYSTEM),
            HumanMessage(content=f"Análisis a evaluar:\n{analysis.model_dump_json()}")
        ])
        return {"evaluation": evaluation}
    except (ValidationError, Exception):
        return {"evaluation": FilmEvaluation(
            completeness_score=5.0,
            has_specific_examples=False,
            improvement_suggestion="No se pudo evaluar automáticamente."
        )}

# Construir grafo
builder = StateGraph(AnalysisState)
builder.add_node("extractor", node_extractor)
builder.add_node("evaluador", node_evaluador)

builder.add_edge(START, "extractor")
builder.add_edge("extractor", "evaluador")
builder.add_edge("evaluador", END)

graph = builder.compile()
print("Grafo de 2 nodos estructurados compilado ✓")

In [ ]:
result = graph.invoke({
    "messages": [HumanMessage(content="¿Cómo usa Nolan el tiempo en Interstellar?")],
    "film_analysis": None,
    "evaluation": None,
    "query": "tiempo en Interstellar"
})

print("=== Análisis Estructurado ===")
rprint(result["film_analysis"])

print("\n=== Evaluación ===")
rprint(result["evaluation"])

## Resumen del Capítulo

| Concepto | Descripción |
|---------|-------------|
| `BaseModel` | Define el schema de salida con tipos y validación |
| `with_structured_output(Schema)` | Instruye al LLM a devolver JSON válido para el schema |
| `ValidationError` | Error cuando el LLM devuelve JSON con tipos incorrectos |
| Fallback | `try/except ValidationError` con valor por defecto |
| Cadena de nodos | 2+ nodos estructurados que se pasan objetos Pydantic vía estado |

**Próximo capítulo**: Memoria y checkpointing con MemorySaver